In [ ]:
"""Cell 1 — Setup, imports, and loading the three real test corpora.

Trains an Electra-large scalar-mix classifier on each of three LLM-generated
synthetic Q/A datasets (Claude, Mistral, DeepSeek) and evaluates each
trained model against three real corpora (ScienceQA, OneStopEnglish, RACE).
The full training and testing transcript for each run is written to its
own .txt log so the per-epoch losses and classification reports are
preserved.
"""
from google.colab import drive
drive.mount("/content/drive")

import os
import gc
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.utils import shuffle
from allennlp.modules.scalar_mix import ScalarMix
from transformers import AutoTokenizer, AutoModel
import transformers
from tqdm import tqdm
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RNG_SEED = 1
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)

label_names = ["elementary", "middle", "high"]
LEVEL_TO_INT = {"elementary": 0, "middle": 1, "high": 2}
INT_TO_LEVEL = {v: k for k, v in LEVEL_TO_INT.items()}

# ---- Per-model synthetic CSVs --------------------------------------------
SYNTH_CSVS = {
    "claude":   "/content/drive/MyDrive/multi_corpus/synthetic_claude_qa.csv",
    "mistral":  "/content/drive/MyDrive/multi_corpus/synthetic_mistral_qa.csv",
    "deepseek": "/content/drive/MyDrive/multi_corpus/synthetic_deepseek_qa.csv",
}

# ---- Test corpora --------------------------------------------------------
def grade_to_label(grade):
    if isinstance(grade, str):
        digits = "".join(filter(str.isdigit, grade))
        if not digits: return None
        num = int(digits)
    else:
        num = int(grade)
    if 1 <= num <= 5:  return 0
    if 6 <= num <= 8:  return 1
    if 9 <= num <= 12: return 2
    return None


def prepare_scienceqa(split_data):
    texts, labels = [], []
    for row in split_data:
        label = grade_to_label(row.get("grade"))
        if label is None: continue
        question = row["question"]
        choices = row["choices"]
        answer_idx = row["answer"]
        answer_txt = choices[answer_idx] if answer_idx < len(choices) else ""
        lecture = row.get("lecture", "") or ""
        solution = row.get("solution", "") or ""
        text = (
            f"Question: {question}\n"
            f"        Choices: {', '.join(f'{chr(65+i)}) {c}' for i, c in enumerate(choices))}\n"
            f"        Correct Answer: {chr(65+answer_idx)}) {answer_txt}\n"
            f"        Explanation: {lecture}\n"
            f"        Solution: {solution}"
        )
        texts.append(text)
        labels.append(label)
    return np.array(texts), np.array(labels)


print("Loading ScienceQA …")
_sqa = load_dataset("tasksource/ScienceQA_text_only")
_sqa_X_tr, _sqa_y_tr = prepare_scienceqa(_sqa["train"])
_sqa_X_te, _sqa_y_te = prepare_scienceqa(_sqa["test"])
SQA_X = np.concatenate([_sqa_X_tr, _sqa_X_te])
SQA_y = np.concatenate([_sqa_y_tr, _sqa_y_te])
print(f"  ScienceQA: {len(SQA_X)} samples; class counts = {np.bincount(SQA_y)}")

print("Loading OneStopEnglish …")
_ose = load_dataset("iastate/onestop_english")["train"]
OSE_X = [row["text"] for row in _ose]
OSE_y = np.array([int(row["label"]) for row in _ose])
print(f"  OneStopEnglish: {len(OSE_X)} samples; class counts = {np.bincount(OSE_y)}")

RACE_PATHS = {
    "race-middle": "/content/drive/MyDrive/multi_corpus/ood_race-middle.csv",
    "race-high":   "/content/drive/MyDrive/multi_corpus/ood_race-high.csv",
}
for _n, _p in list(RACE_PATHS.items()):
    if not os.path.exists(_p) and os.path.exists(os.path.basename(_p)):
        RACE_PATHS[_n] = os.path.basename(_p)

_race_dfs = []
for _name, _path in RACE_PATHS.items():
    if not os.path.exists(_path):
        print(f"  [skip] {_name}: not found at {_path}")
        continue
    _d = pd.read_csv(_path)
    _d = _d.dropna(subset=["full_text", "education_level"]).reset_index(drop=True)
    _d = _d[_d["education_level"].isin(LEVEL_TO_INT)].reset_index(drop=True)
    _race_dfs.append(_d)
    print(f"  loaded {_name}: {len(_d)} rows")

if _race_dfs:
    _race = pd.concat(_race_dfs, ignore_index=True)
    RACE_X = _race["full_text"].astype(str).tolist()
    RACE_y = _race["education_level"].map(LEVEL_TO_INT).values
    print(f"  RACE combined: {len(RACE_X)} samples; class counts = {np.bincount(RACE_y, minlength=3)}")
else:
    RACE_X, RACE_y = None, None


In [ ]:
"""Cell 2 — Scalar-mixed transformer encoder model.

Same architecture as the original BEA-1.40 setup: AutoModel base →
ScalarMix over hidden layers → mean-pool → 2-layer ReLU head → logits.
The `_log` method writes to stdout AND appends to `self.log_path`,
so every per-epoch loss and held-in classification report is preserved.
"""


class ScoringModel(torch.nn.Module):
    def __init__(self, language_model, prefix, num_classes=3) -> None:
        super().__init__()
        self.prefix = prefix
        self.tokenizer = AutoTokenizer.from_pretrained(language_model)
        self.lm = AutoModel.from_pretrained(language_model).to(device)
        self.scalar_mix = ScalarMix(self.lm.config.num_hidden_layers + 1)
        self.dropout = torch.nn.Dropout(p=0.2)
        self.lm_name = language_model

        self.classification_head = torch.nn.Sequential(
            torch.nn.Linear(self.lm.config.hidden_size, self.lm.config.hidden_size),
            torch.nn.ReLU(),
            torch.nn.Linear(self.lm.config.hidden_size, num_classes),
        )
        self.loss = torch.nn.CrossEntropyLoss()

        self.X = None
        self.y = None
        self.eval_X = None
        self.eval_y = None
        self.log_path = None

    def set_dataset(self, X, y):
        self.X = X
        self.y = y

    def set_evalset(self, X, y):
        self.eval_X = X
        self.eval_y = y

    def _log(self, msg=""):
        print(msg)
        if self.log_path is not None:
            with open(self.log_path, "a") as f:
                f.write(str(msg) + "\n")

    def self_eval(self):
        self.eval()
        predictions = []
        with torch.no_grad():
            for text in tqdm(self.eval_X):
                logits = self.forward(text)
                predictions.append(logits.argmax(dim=1).item())
        true_labels = self.eval_y
        if torch.is_tensor(true_labels):
            true_labels = true_labels.cpu().numpy()
        report = classification_report(true_labels, predictions, target_names=label_names)
        self._log(report)
        return {"macro_f1": f1_score(true_labels, predictions, average="macro")}

    def forward(self, input):
        inputs = self.tokenizer(
            input, return_tensors="pt", padding=True, truncation=True,
            max_length=self.lm.config.max_position_embeddings - 2,
        )
        outputs = self.lm(**inputs.to(device), output_hidden_states=True)
        hidden_states = outputs.hidden_states
        result = self.classification_head(
            torch.mean(self.dropout(self.scalar_mix(hidden_states)), dim=1)
        )
        return result

    def fit(self, epochs, optimizer, scheduler, batch_size=4) -> None:
        self.train()
        for epoch in range(epochs):
            self._log(f"Epoch {epoch}")
            r = 0.0
            num_s = 0.0
            d, d_y = shuffle(self.X, self.y)
            batches_X = [d[n:n + batch_size]   for n in range(0, len(d),   batch_size)]
            batches_y = [d_y[n:n + batch_size] for n in range(0, len(d_y), batch_size)]
            for batch in tqdm(range(len(batches_X))):
                pred = self.forward(list(batches_X[batch]))
                ls = self.loss(pred, batches_y[batch])
                optimizer.zero_grad()
                ls.backward()
                optimizer.step()
                scheduler.step()
                r += ls.detach().item()
                num_s += 1
                if batch % 10 == 0 and batch > 0:
                    print(str(r / num_s))
            self._log(f"  Epoch {epoch}: avg loss {r / max(num_s, 1):.4f}")
            if self.eval_X is not None:
                ev = self.self_eval()
                self._log(ev)
                self.train()


In [ ]:
"""Cell 3 — Reusable train+eval driver.

`run_experiment(synth_csv, tag)` runs the full pipeline for one LLM:
  - load the synthetic CSV
  - 80/20 stratified split for held-in monitoring
  - train Electra-large for 3 epochs (per-epoch loss + held-in
    classification report logged)
  - evaluate on ScienceQA, OneStopEnglish, RACE-combined
  - write everything to electra_<tag>_full_log.txt and a per-tag CSV
"""


def predict_with(m, texts):
    m.eval()
    preds = []
    with torch.no_grad():
        for t in tqdm(texts, desc="predict"):
            logits = m.forward(str(t))
            preds.append(logits.argmax(dim=1).item())
    return preds


def run_experiment(synth_csv: str, tag: str):
    log_path = f"electra_{tag}_full_log.txt"
    open(log_path, "w").close()  # truncate

    def log(msg=""):
        print(msg)
        with open(log_path, "a") as f:
            f.write(str(msg) + "\n")

    # ---- Load synthetic CSV ----
    if not os.path.exists(synth_csv):
        local = os.path.basename(synth_csv)
        if os.path.exists(local):
            synth_csv = local
        else:
            raise FileNotFoundError(f"Synthetic CSV not found: {synth_csv}")
    s = pd.read_csv(synth_csv)
    log(f"\n{'#'*60}\n# Run: {tag}\n# Source: {synth_csv}\n{'#'*60}")
    log(f"[{tag}] Loaded {len(s)} synthetic rows")
    log(f"[{tag}] Class distribution: {s['grade_level'].value_counts().to_dict()}")

    s = s.dropna(subset=["question", "answer", "grade_level"]).reset_index(drop=True)
    s = s[s["grade_level"].isin(LEVEL_TO_INT)].reset_index(drop=True)

    X_all = np.array([f"Question: {r['question']}\n        Answer: {r['answer']}"
                      for _, r in s.iterrows()])
    y_all = s["grade_level"].map(LEVEL_TO_INT).values

    X_tr, X_he, y_tr, y_he = train_test_split(
        X_all, y_all, test_size=0.20, stratify=y_all, random_state=RNG_SEED,
    )
    log(f"[{tag}] Train: {len(X_tr)}   Held-in: {len(X_he)}")

    # ---- Train ----
    y_tr_t = torch.tensor(y_tr, dtype=torch.long).to(device)
    y_he_t = torch.tensor(y_he, dtype=torch.long).to(device)

    m = ScoringModel("google/electra-large-discriminator", "run", num_classes=3).to(device)
    m.set_dataset(X_tr, y_tr_t)
    m.set_evalset(X_he, y_he_t)
    m.log_path = log_path   # ScoringModel._log writes here

    optimizer = torch.optim.AdamW(m.parameters(), lr=1e-5)
    scheduler = transformers.get_cosine_schedule_with_warmup(
        optimizer=optimizer, num_warmup_steps=50,
        num_training_steps=3 * len(X_tr),
    )
    m.fit(3, optimizer, scheduler=scheduler, batch_size=8)
    torch.save(m.state_dict(), f"final-model-electra-{tag}.pt")

    # ---- Evaluate ----
    def eval_corpus(name, X, y, labels_present):
        log(f"\n{'='*60}\n[{tag}] Eval: {name}\n{'='*60}")
        preds = predict_with(m, X)
        acc = accuracy_score(y, preds)
        f1m = f1_score(y, preds, labels=labels_present, average="macro", zero_division=0)
        pred_counts = Counter(preds)
        log(f"  Accuracy: {acc:.4f}")
        log(f"  Macro-F1 over {labels_present}: {f1m:.4f}")
        log(f"  Prediction distribution: "
            f"{{ {', '.join(f'{INT_TO_LEVEL[k]}: {v}' for k, v in sorted(pred_counts.items()))} }}")
        union = sorted(set(list(y) + list(preds)))
        log(classification_report(
            y, preds, labels=union,
            target_names=[INT_TO_LEVEL[i] for i in union],
            digits=3, zero_division=0,
        ))
        return {"name": name, "accuracy": acc, "macro_f1": f1m}

    rows = []
    rows.append(eval_corpus("ScienceQA (all)", SQA_X, SQA_y, [0, 1, 2]))
    rows.append(eval_corpus("OneStopEnglish",  OSE_X, OSE_y, [0, 1, 2]))
    if RACE_X is not None:
        rows.append(eval_corpus("RACE combined (middle+high)", RACE_X, RACE_y, [1, 2]))

    summary = pd.DataFrame([
        {"trained_on": tag, "corpus": r["name"],
         "accuracy": r["accuracy"], "macro_f1": r["macro_f1"]}
        for r in rows
    ])
    out_csv = f"synthetic_eval_summary_{tag}.csv"
    summary.to_csv(out_csv, index=False)
    log(f"\n[{tag}] Summary:")
    log(summary.to_string(index=False))
    log(f"\n[{tag}] Saved CSV → {out_csv}")
    log(f"[{tag}] Full log  → {log_path}")

    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


In [ ]:
"""Cell 4 — Run the experiment for all three LLMs."""
all_summaries = []
for tag, csv_path in SYNTH_CSVS.items():
    print(f"\n\n{'#'*70}")
    print(f"#   Starting run for: {tag}")
    print(f"{'#'*70}")
    summary = run_experiment(csv_path, tag)
    all_summaries.append(summary)


In [ ]:
"""Cell 5 — Combined comparison across the three LLMs.

Stacks the per-tag summaries and pivots them into accuracy/F1 tables
where rows are test corpora and columns are the training LLM.
"""
combined = pd.concat(all_summaries, ignore_index=True)

pivot_acc = combined.pivot(index="corpus", columns="trained_on", values="accuracy")
pivot_f1  = combined.pivot(index="corpus", columns="trained_on", values="macro_f1")

print("\n=== Accuracy (rows=test corpus, cols=trained-on LLM) ===")
print(pivot_acc.round(4).to_string())
print("\n=== Macro-F1 (rows=test corpus, cols=trained-on LLM) ===")
print(pivot_f1.round(4).to_string())

combined.to_csv("synthetic_eval_summary_all_llms.csv", index=False)
pivot_acc.to_csv("synthetic_eval_pivot_accuracy.csv")
pivot_f1.to_csv("synthetic_eval_pivot_macrof1.csv")
print("\nSaved → synthetic_eval_summary_all_llms.csv")
print("Saved → synthetic_eval_pivot_accuracy.csv")
print("Saved → synthetic_eval_pivot_macrof1.csv")
